<div align="right"><a href="S32-PSD-Estimation-using-the-Bartlett-Welch-Method.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

Dieses Jupyter Notebook kann durch Klicken auf das Symbol rechts in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) geöffnet werden. Am besten wird es jedoch lokal mit [Jupyter Notebook](https://jupyter.org/) angezeigt (z. B. als Teil der [Anaconda](https://www.anaconda.com/)-Distribution). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/python/L32-Schaetzung-des-Leistungsdichtespektrums-nach-Bartlett-und-Welch.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Zurück: [L31 Schätzung des Leistungsdichtespektrums nach Blackman-Tuckey](./L31-Schaetzung-des-Leistungsdichtespektrums-nach-Blackman-Tuckey.ipynb) | [Aufgabe 32](./A32-Schaetzung-des-Leistungsdichtespektrums-nach-Bartlett-und-Welch.ipynb) | [Inhalt / Übersicht](./00-Python-Notebooks-Overview-DE.ipynb) | Weiter: [L33 Yule-Walker-/Autokorrelationsmethode](./L33-Yule-Walker-und-Autokorrelationsmethode.ipynb)

</div>

---


## Aufgabe: Klassische Spektralschätzung — Bartlett & Welch (8.4)

Um die Varianz des Periodogramms  
$\mathrm{Per}_N^X(\Omega)$ zu verringern, berechnen sowohl der
Bartlett- als auch der Welch-Algorithmus den Mittelwert mehrerer
($K$) Periodogramme, die aus Teilsequenzen der Länge

$$
L = \frac{N}{K}
$$

gewonnen werden.

Die Welch-Methode erweitert den Bartlett-Algorithmus, indem zusätzlich jede Teilsequenz mit einer Fensterfunktion gewichtet wird. Der Bartlett- Algorithmus ist als Spezialfall in der Welch-Methode enthalten, wenn ein rechteckiges Fenster verwendet wird.

Vergleichen Sie die Realisierungen beider stochastischer Prozesse der Länge

$$
N = 2^{10}
$$

aus Aufgabe `dsv8a02e`.

Analysieren Sie den Einfluss von  
- unterschiedlicher Anzahl an Periodogrammen zur Mittelwertbildung  
  ($K = 1, 4, 16, 64$), sowie  
- verschiedenen Fensterfunktionen (rechteckig, Hamming)  

auf die Qualität der Spektralschätzung.

---

### Python-Hinweise

```python
# Spektralschätzung mit Welch (aus helper_functions)
S_xx_est = lwelch(data, k, window_type, nfft, overlap)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter
from helper_functions import lwelch
# BEGIN SOLUTION
def run_estimation_plots(data, true_psd, axis_limits, method_name, window_type):
    """
    Helper function to generate plots for different segment counts (v_K).

    Parameters:
    -----------
    data : ndarray
        The filtered signal to analyze.
    true_psd : ndarray
        The theoretically calculated PSD for comparison.
    axis_limits : list
        Plot limits in the format [xmin, xmax, ymin, ymax].
    method_name : str
        Name of the method for the plot title (e.g., 'Bartlett', 'Welch').
    window_type : str
        The window string identifier for the lwelch function.
    """
    v_k = [1, 4, 16, 64]
    n_samples = 2**10  

    for k in v_k:
        s_xx_est = lwelch(data, k, window_type, nfft=2**10, overlap=False)
        
        plt.figure(figsize=(9, 5))
        plt.plot(f_t, s_xx_est, label='Estimate', color='tab:blue')
        plt.plot(f_t, true_psd, '--', label='True PSD', color='tab:orange')
        
        plt.axis(axis_limits)
        plt.xlabel(r'$\Omega/2\pi = f \cdot T$')
        plt.ylabel(r'$S_{XX}(\exp(j \cdot \Omega))$')
        
        title = f"{method_name} Estimate of $S_{{XX}}$; K={k} and L={n_samples//k}"
        if window_type == 'hamming':
            title += " (Hamming)"
        plt.title(title)
        
        if k == 1:
            plt.text(0.73, axis_limits[3] * 0.6, '== Periodogram')
            
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.legend()
        plt.show()

np.random.seed(42)  
N = 2**10           # Signal length
N_PRE = 500         # Transient/Pre-run length
NFFT = 2**10        # FFT points

noise = np.random.randn(N + N_PRE)

# FIR Filter for x1
h1 = [1, -4, 6, -4, 1]
x1_full = lfilter(h1, 1, noise)
x1 = x1_full[N_PRE:]

# IIR Filter for x2
h2 = [1, 0.8]
x2_full = lfilter([1], h2, noise)
x2 = x2_full[N_PRE:]

f_t = np.linspace(0, 1, NFFT, endpoint=False)

sx1x1_true = np.abs(np.fft.fft(h1, NFFT))**2
axis_sx1 = [0, 1, 0, 1.5 * np.max(sx1x1_true)]

sx2x2_true = 1.0 / np.abs(np.fft.fft(h2, NFFT))**2
axis_sx2 = [0, 1, 0, 1.5 * np.max(sx2x2_true)]

print("Generating Bartlett plots for x1...")
run_estimation_plots(x1, sx1x1_true, axis_sx1, "Bartlett", 'boxcar')

print("Generating Welch plots for x1 (Hamming)...")
run_estimation_plots(x1, sx1x1_true, axis_sx1, "Welch", 'hamming')

print("Generating Bartlett plots for x2...")
run_estimation_plots(x2, sx2x2_true, axis_sx2, "Bartlett", 'boxcar')

print("Generating Welch plots for x2 (Hamming)...")
run_estimation_plots(x2, sx2x2_true, axis_sx2, "Welch", 'hamming')
# END SOLUTION

# Lösung: Klassische Spektralschätzung — Bartlett & Welch (8.4)

Die folgenden Abbildungen zeigen die geschätzte Leistungsdichtespektren $S_{XX}^{W}$, basierend auf

$$
N = 2^{10} = 1024
$$

Datenwerten der Stichprobenprozesse $X_1(k)$ und $X_2(k)$, berechnet mit dem Bartlett- und dem Welch-Algorithmus (Hamming-Fenster).

Die gemittelten Periodogramme werden aus Teilsegmenten des Datensatzes mit den Längen

$$
L = \frac{N}{K} = 1024,\; 256,\; 64,\; \text{und } 16
$$

Samples berechnet.

Die wahre Leistungsdichtespektren $S_{X_i X_i}(e^{j\Omega})$ sind jeweils durch eine gestrichelte Linie dargestellt. Es ist zu erkennen, dass durch das Mitteln einer ausreichend großen Anzahl von Periodogrammen eine akzeptable Spektralschätzung erreicht wird.

---

## Stichprobenprozess $X_1(k)$

### Bartlett

<div>
  <img src="./pngs/ldsv8a04_x1barK1.png"/>
  <img src="./pngs/ldsv8a04_x1barK4.png"/>
  <img src="./pngs/ldsv8a04_x1barK16.png"/>
  <img src="./pngs/ldsv8a04_x1barK64.png"/>
</div>

### Welch

<div>
  <img src="./pngs/ldsv8a04_x1welK1.png"/>
  <img src="./pngs/ldsv8a04_x1welK4.png"/>
  <img src="./pngs/ldsv8a04_x1welK16.png"/>
  <img src="./pngs/ldsv8a04_x1welK64.png"/>
</div>

---

## Stichprobenprozess $X_2(k)$

### Bartlett

<div>
  <img src="./pngs/ldsv8a04_x2barK1.png"/>
  <img src="./pngs/ldsv8a04_x2barK4.png"/>
  <img src="./pngs/ldsv8a04_x2barK16.png"/>
  <img src="./pngs/ldsv8a04_x2barK64.png"/>
</div>

### Welch

<div>
  <img src="./pngs/ldsv8a04_x2welK1.png"/>
  <img src="./pngs/ldsv8a04_x2welK4.png"/>
  <img src="./pngs/ldsv8a04_x2welK16.png"/>
  <img src="./pngs/ldsv8a04_x2welK64.png"/>
</div>